# Maritime VHF ASR — Dissertation Notebook
Transformer-based ASR fine-tuning on real and simulated maritime VHF speech.

**Before you start:** Runtime → Change runtime type → T4 GPU

| Model | GPU |
|---|---|
| Whisper-small / Wav2Vec2 | T4 (free tier) |
| Whisper-medium | T4 (~30 min / experiment) |
| Whisper-large + LoRA | A100 (Colab Pro+) |

## Step 1 — GPU & Drive

In [ ]:
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU : {gpu.name}  ({gpu.total_memory/1e9:.1f} GB VRAM)")
else:
    print("No GPU detected — Runtime → Change runtime type → T4 GPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Sync Project from GitHub

Run this at the start of every session and after pushing local changes.

In [ ]:
import os

REPO    = "https://github.com/tolgaSarmi/maritime_asr.git"
WORKDIR = "/content/asr_dissertation"

if os.path.exists(WORKDIR):
    !git -C {WORKDIR} pull
else:
    !git clone {REPO} {WORKDIR}

os.chdir(WORKDIR)
print("Working directory:", os.getcwd())

### GitHub workflow

Push changes from your local machine, then re-run the cell above to pull them into Colab.

**Persistence:**
- Checkpoints auto-save to Drive every 200 steps → `/content/drive/MyDrive/ASR_Dissertation/checkpoints/`
- Manifests and results also live on Drive
- The `/content/asr_dissertation` directory is ephemeral and is recreated by Step 2

## Step 3 — Install Dependencies

Run **Cell A**, then **Runtime → Restart session**, then **Cell B** and **Cell C**.

In [ ]:
# Cell A — install, then restart the session
!pip install -q \
    "scipy>=1.14.0" \
    "scikit-learn>=1.6.0" \
    "torchao>=0.16.0" \
    transformers peft datasets accelerate evaluate \
    jiwer librosa soundfile omegaconf rich inflect \
    tensorboard seaborn
print("Done — Runtime → Restart session now")

In [ ]:
# Cell B — run immediately after restart
import os
os.chdir('/content/asr_dissertation')
print("Working directory:", os.getcwd())

In [ ]:
# Cell C — verify the environment
import numpy, scipy, transformers, peft, omegaconf, evaluate, torchao
from packaging.version import Version

print(f"numpy        : {numpy.__version__}")
print(f"scipy        : {scipy.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print(f"torchao      : {torchao.__version__}")

# torchao >= 0.16.0 is required by PEFT 0.19+ for LoRA model loading
if Version(torchao.__version__) < Version("0.16.0"):
    raise RuntimeError(
        f"torchao {torchao.__version__} is too old — re-run Cell A, restart, then Cell B and C."
    )

print("Environment OK")

## Step 4 — Label Studio API Key & Data Export

**Real dataset** — audio streams from Azure; only metadata and URLs are downloaded.  
**Simulated dataset** — audio files are downloaded to disk from a public Azure container.

In [ ]:
import os
try:
    from google.colab import userdata
    api_key = userdata.get('LABEL_STUDIO_API_KEY')
    print('API key loaded from Colab Secrets')
except Exception:
    api_key = ''  # paste key here if not using Secrets
    if not api_key:
        print('No API key — add LABEL_STUDIO_API_KEY to Colab Secrets or paste it above')

os.environ['LABEL_STUDIO_API_KEY'] = api_key

In [ ]:
# Real dataset: fetch task metadata and Azure audio URLs (no audio download)
!python label_studio_export.py --api-key $LABEL_STUDIO_API_KEY --project real

# Simulated dataset: download audio files to disk
!python label_studio_export.py --api-key $LABEL_STUDIO_API_KEY --project simulated

In [ ]:
# Alternative to the export above: restore manifests from Drive (faster in future sessions)
DRIVE_PROJECT = '/content/drive/MyDrive/ASR_Dissertation'
import shutil, os, glob

manifests = glob.glob(f'{DRIVE_PROJECT}/data/**/*.json', recursive=True)
if not manifests:
    print('No saved manifests on Drive — run the export cells above first')
else:
    for src in manifests:
        rel = os.path.relpath(src, f'{DRIVE_PROJECT}/')
        os.makedirs(os.path.dirname(rel), exist_ok=True)
        shutil.copy(src, rel)
        print(f'Restored: {rel}')

In [ ]:
# Save manifests to Drive after the first successful export so future sessions can restore them
DRIVE_PROJECT = '/content/drive/MyDrive/ASR_Dissertation'
import shutil, os, glob

os.makedirs(f'{DRIVE_PROJECT}/data', exist_ok=True)
for manifest in glob.glob('data/**/*.json', recursive=True):
    dst = f'{DRIVE_PROJECT}/{manifest}'
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(manifest, dst)
    print(f'Saved: {manifest}')

## Step 5 — Prepare Datasets

In [ ]:
# Validate audio, normalise transcriptions, create 80/10/10 train/val/test splits
!python main.py --mode data

In [ ]:
import json, glob
for path in sorted(glob.glob('data/*/*.json')):
    with open(path) as f:
        n = len(json.load(f))
    print(f"{path:<50} {n:>5} samples")

## Step 6 — Training

If Colab disconnects mid-run, re-run the same experiment — training resumes automatically from the latest checkpoint.

Start with `ef_whisper_small_real` to verify your setup (~30 min on T4), then run the batch cell below.

In [ ]:
!python main.py --mode list

In [ ]:
EXPERIMENT = "ef_whisper_small_real"
!python main.py --mode train --experiment {EXPERIMENT}

In [ ]:
# Runs sequentially; completed experiments are skipped automatically.
EXPERIMENTS = [
    "ef_whisper_small_real",
    "ef_whisper_medium_real",
    "ef_wav2vec2_real",
    "lora_whisper_small_real",
    "lora_whisper_medium_real",
    # Uncomment for A100 (Colab Pro+):
    # "ef_whisper_large_real",
    # "lora_whisper_large_real",
    # "full_ft_whisper_medium_real",
    # "full_ft_whisper_large_real",
]

import subprocess, sys
for exp in EXPERIMENTS:
    print(f"\n{'='*55}\n  {exp}\n{'='*55}")
    r = subprocess.run(
        [sys.executable, "main.py", "--mode", "train", "--experiment", exp],
        cwd="/content/asr_dissertation",
    )
    if r.returncode != 0:
        print(f"  {exp} failed — continuing")

In [ ]:
# List all checkpoints currently on Drive
import os

checkpoint_dir = '/content/drive/MyDrive/ASR_Dissertation/checkpoints'
if os.path.exists(checkpoint_dir):
    for exp in sorted(os.listdir(checkpoint_dir)):
        exp_path = os.path.join(checkpoint_dir, exp)
        if os.path.isdir(exp_path):
            ckpts = sorted(d for d in os.listdir(exp_path) if d.startswith('checkpoint-'))
            status = ', '.join(ckpts) if ckpts else 'no checkpoints yet'
            print(f"{exp}: {status}")
else:
    print("No checkpoints on Drive yet — train first")

In [ ]:
import json
from pathlib import Path

EXPERIMENT = "ef_whisper_small_real"
CHECKPOINT = "checkpoint-400"

state_file = Path(f"/content/drive/MyDrive/ASR_Dissertation/checkpoints/{EXPERIMENT}/{CHECKPOINT}/trainer_state.json")
if state_file.exists():
    state = json.load(open(state_file))
    print(f"Step  : {state.get('global_step')}")
    print(f"Epoch : {state.get('epoch', 0):.2f}")
    print(f"Best WER : {state.get('best_metric')}")
else:
    print(f"Not found: {state_file}")

## Step 7 — Evaluate

In [ ]:
!python main.py --mode eval_all

In [ ]:
CHECKPOINT = "/content/drive/MyDrive/ASR_Dissertation/checkpoints/ef_whisper_small_real"
MANIFEST   = "data/real/test_manifest.json"
!python main.py --mode test_wer --checkpoint {CHECKPOINT} --manifest {MANIFEST}

## Step 8 — Figures

In [ ]:
!python main.py --mode figures

In [ ]:
from IPython.display import Image, display
import glob

for fig in sorted(glob.glob('results/figures/*.png')):
    print(f"\n{fig.split('/')[-1]}")
    display(Image(fig, width=900))

In [ ]:
import shutil
DRIVE_PROJECT = '/content/drive/MyDrive/ASR_Dissertation'
shutil.copytree('/content/asr_dissertation/results', f'{DRIVE_PROJECT}/results', dirs_exist_ok=True)
print("Results saved to Drive")

## Step 9 — Results Table

In [ ]:
import json, pandas as pd

try:
    with open('results/all_results.json') as f:
        all_results = json.load(f)

    rows = []
    for exp_name, res in all_results.items():
        for key, val in res.items():
            if key.startswith('eval_') and isinstance(val, dict) and 'wer' in val:
                rows.append({
                    'Experiment': exp_name,
                    'Method'    : res.get('method', ''),
                    'Train Data': str(res.get('train_data', '—')),
                    'Eval Set'  : key.replace('eval_', ''),
                    'WER (%)'   : round(val['wer'] * 100, 2),
                    'CER (%)'   : round(val.get('cer', 0) * 100, 2),
                })

    df = pd.DataFrame(rows).sort_values(['Method', 'Eval Set', 'WER (%)'])
    pd.set_option('display.max_rows', 100)
    print(df.to_string(index=False))

except FileNotFoundError:
    print("No results yet — run Step 7 first")

## Bonus — Transcribe a Single File

In [ ]:
from src.inference import build_transcriber
from google.colab import files

uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

CHECKPOINT = "checkpoints/ef_whisper_small_real"  # change to the best model
transcriber = build_transcriber("whisper", CHECKPOINT)
text, elapsed = transcriber.transcribe_file(audio_path)

print(f"Transcription : {text}")
print(f"Latency       : {elapsed*1000:.0f} ms")

---
## Session Reset

After Colab disconnects:

1. Step 1 — GPU check and Drive mount
2. Step 2 — pull latest code from GitHub
3. Step 3 — Cell B then Cell C (skip Cell A unless packages changed)
4. Step 4 — restore manifests from Drive (~2 s)
5. Step 5 — re-run data pipeline (~30 s)
6. Step 6 — continue training (auto-resumes from last checkpoint)

---
## Out of Memory?

The config is tuned for T4 (`batch_size: 4`, `gradient_accumulation: 8`, effective batch = 32).
If you still hit OOM on Whisper-large, switch to an A100 runtime or reduce in `configs/config.yaml`:

```yaml
per_device_train_batch_size: 2
gradient_accumulation_steps: 16
```